In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Project_NLP/dataset_final.csv')
df.head()
display(df)

,text,label
0,Cán bộ công an tử vong vì tai nạn giao thông h...,1
1,Chủ tịch UBND TP HCM: Chọn 38 trung tâm giải q...,1
2,Dùng cần cẩu giải cứu 2 người đi xe máy rơi xu...,1
3,Khói bụi trắng bao trùm khu vực gần nhà máy th...,1
4,Công bố quy trình 5 bước bổ nhiệm công chức là...,1
...,...,...
46831,BRUSSELS (Reuters) - Các đồng minh NATO hôm th...,1
46832,"LONDON (Reuters) - LexisNexis, nhà cung cấp th...",1
46833,MINSK (Reuters) - Dưới bóng các nhà máy thời X...,1
46834,MOSCOW (Reuters) - Bộ trưởng Ngoại giao Vatica...,1


In [ ]:
print(df['text'][16])

Lòng tốt luôn ở quanh ta. Trong cuộc sống bộn bề lo toan, những câu chuyện về lòng tử tế luôn là ngọn lửa sưởi ấm tâm hồn và lan tỏa năng lượng tích cực Sự tử tế trong guồng quay hối hả của cuộc sống không chỉ là những hành động lớn lao mà đôi khi là sự quan tâm, sẻ chia với những cảnh đời kém may mắn. Gánh bánh mì yêu thương Cụ Nguyễn Thị Ngang (phường Phú An, TP HCM) được nhiều người trìu mến gọi với cái tên thân thương ngoại Sáu". Gần 40 năm qua


In [ ]:
pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import math
import re
from pyvi import ViTokenizer
from collections import defaultdict
import numpy as np

In [ ]:
import regex as re

from pyvi import ViTokenizer
EMAIL = re.compile(r"([\w0-9_\.-]+)(@)([\d\w\.-]+)(\.)([\w\.]{2,6})")
URL = re.compile(r"https?:\/\/(?!.*:\/\/)\S+")
PHONE = re.compile(r"(09|01[2|6|8|9])+([0-9]{8})\b")
MENTION = re.compile(r"@.+?:")
NUMBER = re.compile(r"\d+.?\d*")
DATETIME = '\d{1,2}\s?[/-]\s?\d{1,2}\s?[/-]\s?\d{4}'

RE_HTML_TAG = re.compile(r'<[^>]+>')
RE_CLEAR_1 = re.compile("[^_<>\s\p{Latin}]")
RE_CLEAR_2 = re.compile("__+")
RE_CLEAR_3 = re.compile("\s+")



def replace_common_token(txt):
    txt = re.sub(EMAIL, ' ', txt)
    txt = re.sub(URL, ' ', txt)
    txt = re.sub(MENTION, ' ', txt)
    txt = re.sub(DATETIME, ' ', txt)
    txt = re.sub(NUMBER, ' ', txt)
    return txt


def remove_emoji(txt):
    txt = re.sub(':v', '', txt)
    txt = re.sub(':D', '', txt)
    txt = re.sub(':3', '', txt)
    txt = re.sub(':\(', '', txt)
    txt = re.sub(':\)', '', txt)
    return txt


def remove_html_tag(txt):
    return re.sub(RE_HTML_TAG, ' ', txt)

def preprocess(txt, tokenize=True):
    txt = remove_html_tag(txt)
    txt = re.sub('&.{3,4};', ' ', txt)

    if tokenize:
        txt = ViTokenizer.tokenize(txt)
    txt = txt.lower()
    txt = replace_common_token(txt)
    txt = remove_emoji(txt)
    txt = re.sub(RE_CLEAR_1, ' ', txt)
    txt = re.sub(RE_CLEAR_2, ' ', txt)
    txt = re.sub(RE_CLEAR_3, ' ', txt)
    return txt.strip()

In [ ]:

listDocs = []
for doc in df['text']:
    listDocs.append(preprocess(str(doc)))

In [ ]:
print(df['label'].value_counts())

label
1    25408
0    21428
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer
import torch
from tqdm import tqdm


In [ ]:

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW  # Sử dụng bản gốc từ PyTorch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

# Dùng GPU nếu có
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dùng RoBERTa đa ngôn ngữ
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd


# Giả sử df đã có:
# df = DataFrame có N dòng, cột cuối là nhãn

# Lấy cột nhãn từ df (giả sử là cột cuối)
labels = df.iloc[:, -1].values  # hoặc df[df.columns[-1]]

# Tạo DataFrame mới
df = pd.DataFrame({
    'text': listDocs,
    'label': labels
})

# df = pd.DataFrame(data)


# Tách train/test
# Tách train/test
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.05, random_state=42)

In [ ]:
class RobertaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoded.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = RobertaDataset(X_train.tolist(), y_train.tolist(), tokenizer)
test_dataset = RobertaDataset(X_test.tolist(), y_test.tolist(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

# model.train()
# for epoch in range(3):
#     total_loss = 0
#     for batch in train_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = model(**batch)
#         loss = outputs.loss

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#     print(f"Epoch {epoch + 1} - Loss: {total_loss / len(train_loader):.4f}")


In [ ]:
# from sklearn.metrics import classification_report, accuracy_score

# for epoch in range(5):
#     model.train()
#     total_loss = 0
#     train_preds = []
#     train_labels = []

#     for batch in train_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = model(**batch)
#         loss = outputs.loss

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#         preds = torch.argmax(outputs.logits, dim=1)
#         train_preds.extend(preds.cpu().numpy())
#         train_labels.extend(batch["labels"].cpu().numpy())

#     train_acc = accuracy_score(train_labels, train_preds)

#     # 👇 Đánh giá trên test sau mỗi epoch
#     model.eval()
#     test_preds, test_labels = [], []

#     with torch.no_grad():
#         for batch in test_loader:
#             labels = batch["labels"].cpu().numpy()
#             batch = {k: v.to(device) for k, v in batch.items()}
#             outputs = model(**batch)
#             preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

#             test_preds.extend(preds)
#             test_labels.extend(labels)

#     test_acc = accuracy_score(test_labels, test_preds)

#     print(f"Epoch {epoch + 1} - Train Loss: {total_loss / len(train_loader):.4f} - Train Acc: {train_acc:.4f} - Test Acc: {test_acc:.4f}")


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

for epoch in range(5):
    # ----- Training -----
    model.train()
    total_loss = 0
    train_preds = []
    train_labels = []

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(batch["labels"].cpu().numpy())

    train_acc = accuracy_score(train_labels, train_preds)

    # ----- Evaluation -----
    model.eval()
    test_preds = []
    test_labels = []
    test_loss = 0

    with torch.no_grad():
        for batch in test_loader:
            labels = batch["labels"].cpu().numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            test_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

            test_preds.extend(preds)
            test_labels.extend(labels)

    test_acc = accuracy_score(test_labels, test_preds)

    print(
        f"Epoch {epoch + 1} - "
        f"Train Loss: {total_loss / len(train_loader):.4f} - "
        f"Train Acc: {train_acc:.4f} - "
        f"Test Loss: {test_loss / len(test_loader):.4f} - "
        f"Test Acc: {test_acc:.4f}"
    )

    # Nếu muốn xem chi tiết hơn từng lớp
    # print(classification_report(test_labels, test_preds))


Epoch 1 - Train Loss: 0.1516 - Train Acc: 0.9356 - Test Loss: 0.0930 - Test Acc: 0.9622
Epoch 2 - Train Loss: 0.0862 - Train Acc: 0.9643 - Test Loss: 0.1000 - Test Acc: 0.9611
Epoch 3 - Train Loss: 0.0665 - Train Acc: 0.9729 - Test Loss: 0.0749 - Test Acc: 0.9716
Epoch 4 - Train Loss: 0.0476 - Train Acc: 0.9804 - Test Loss: 0.0799 - Test Acc: 0.9702
Epoch 5 - Train Loss: 0.0373 - Train Acc: 0.9862 - Test Loss: 0.1159 - Test Acc: 0.9678


In [ ]:
# Sau khi model.train() xong
save_path = "/content/drive/MyDrive/roberta_fakenews.pth"

torch.save(model.state_dict(),save_path)

In [ ]:
model.save_pretrained("./roberta_fakenews_model")
tokenizer.save_pretrained("./roberta_fakenews_model")


NameError: name 'model' is not defined